# 🧪 Évaluation RAGAS — BVMT Intelligence

Ce notebook évalue la qualité du système RAG selon les métriques RAGAS :
- **Faithfulness** : La réponse est-elle fondée sur le contexte récupéré ?
- **Answer Relevancy** : La réponse répond-elle à la question ?
- **Context Precision** : Le contexte récupéré est-il précis et pertinent ?
- **Context Recall** : Le contexte récupéré couvre-t-il suffisamment la ground truth ?

In [ ]:
# ─── Installation des dépendances (si nécessaire) ─────────────────────────────
# !pip install ragas langchain-groq langchain-community python-dotenv

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Ajouter le dossier parent au path pour importer les modules du projet
sys.path.insert(0, os.path.abspath('..'))

from dotenv import load_dotenv
load_dotenv('../.env')

print('✅ Environnement chargé.')
print(f"GROQ_API_KEY défini : {'Oui' if os.getenv('GROQ_API_KEY') else 'NON ❌'}")

## 1. Initialisation du Retriever et du Pipeline

In [ ]:
from src.retrieval.hybrid_retriever import get_hybrid_retriever
from src.agents.graph import WorkflowManager

# Utiliser le modèle léger pour accélérer l'évaluation
MODEL_CHOICE = 'light'

print(f'Chargement du retriever (modèle: {MODEL_CHOICE})...')
retriever = get_hybrid_retriever(model_choice=MODEL_CHOICE, use_multi_query=False)

if retriever:
    manager = WorkflowManager(retriever)
    workflow = manager.build_graph()
    print('✅ Pipeline RAG initialisé.')
else:
    print('❌ Retriever non disponible. Lancez d\'abord l\'ingestion.')

## 2. Jeu de données d'évaluation

6 questions couvrant différents types : factuelle, comparative, résumé, tendance, performance, investissement.

In [ ]:
# Dataset d'évaluation : (question, ground_truth)
eval_dataset = [
    {
        "question": "Quelle est la performance du Tunindex en 2022 ?",
        "ground_truth": "Le Tunindex a enregistré une hausse significative en 2022, portée notamment par le secteur bancaire et les valeurs industrielles."
    },
    {
        "question": "Quelles sont les banques les plus performantes à la BVMT ?",
        "ground_truth": "La BIAT, Attijari Bank et Amen Bank figurent parmi les banques les plus performantes à la BVMT, avec des PNB et des résultats nets en croissance."
    },
    {
        "question": "Donne-moi un résumé de la BVMT et de son fonctionnement.",
        "ground_truth": "La BVMT (Bourse des Valeurs Mobilières de Tunis) est le marché financier tunisien où sont cotées des actions, des obligations et d'autres titres financiers."
    },
    {
        "question": "Comment a évolué le secteur bancaire tunisien entre 2020 et 2023 ?",
        "ground_truth": "Le secteur bancaire tunisien a connu une reprise progressive après la crise Covid-19, avec une amélioration des PNB et une baisse progressive du taux de créances classées."
    },
    {
        "question": "Comparer les performances de la SFBT et de la BIAT.",
        "ground_truth": "La SFBT est un leader dans l'agroalimentaire tunisien tandis que la BIAT est la première banque privée de Tunisie. Leurs secteurs d'activité, niveaux de rentabilité et indicateurs financiers diffèrent."
    },
    {
        "question": "Quelles sont les tendances du marché obligataire tunisien ?",
        "ground_truth": "Le marché obligataire tunisien est dominé par les emprunts d'État. Les taux d'intérêt ont augmenté ces dernières années en lien avec la politique monétaire de la BCT."
    },
]

print(f'✅ {len(eval_dataset)} questions chargées.')

## 3. Génération des réponses et des contextes

In [ ]:
from tqdm import tqdm

results = []

for item in tqdm(eval_dataset, desc='Génération des réponses'):
    question = item['question']
    
    initial_state = {
        'question': question,
        'chat_history': [],
        'iterations': 0,
        'documents': [],
        'generation': '',
        'error': '',
        'query_type': ''
    }
    
    try:
        final_state = workflow.invoke(initial_state)
        answer = final_state.get('generation', '')
        docs = final_state.get('documents', [])
        contexts = [doc.page_content for doc in docs]
    except Exception as e:
        print(f'❌ Erreur pour "{question}": {e}')
        answer = ''
        contexts = []
    
    results.append({
        'question': question,
        'answer': answer,
        'contexts': contexts,
        'ground_truth': item['ground_truth']
    })

print(f'\n✅ {len(results)} réponses générées.')

## 4. Affichage des réponses générées

In [ ]:
import pandas as pd

preview_df = pd.DataFrame([{
    'Question': r['question'],
    'Réponse (extrait)': r['answer'][:200] + '...' if len(r['answer']) > 200 else r['answer'],
    'Nb contextes': len(r['contexts'])
} for r in results])

pd.set_option('display.max_colwidth', 120)
display(preview_df)

## 5. Évaluation avec RAGAS

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

# Préparer le dataset RAGAS
ragas_data = {
    'question': [r['question'] for r in results],
    'answer':   [r['answer']   for r in results],
    'contexts': [r['contexts'] for r in results],
    'ground_truth': [r['ground_truth'] for r in results]
}

ragas_dataset = Dataset.from_dict(ragas_data)

# Configurer le LLM et les embeddings pour RAGAS
llm_for_ragas = ChatGroq(model='llama-3.3-70b-versatile', temperature=0)
embeddings_for_ragas = HuggingFaceEmbeddings(
    model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
    model_kwargs={'device': 'cpu'}
)

print('🔄 Évaluation RAGAS en cours (peut prendre quelques minutes)...')

ragas_result = evaluate(
    dataset=ragas_dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ],
    llm=llm_for_ragas,
    embeddings=embeddings_for_ragas
)

print('\n✅ Évaluation terminée !')

## 6. Résultats par question

In [ ]:
results_df = ragas_result.to_pandas()

# Afficher les colonnes pertinentes
cols = ['question', 'faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']
display(results_df[cols].round(3))

## 7. Scores globaux et synthèse

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

metrics = ['faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']
labels = ['Faithfulness', 'Answer\nRelevancy', 'Context\nPrecision', 'Context\nRecall']
scores = [results_df[m].mean() for m in metrics]

print('\n📊 SCORES GLOBAUX RAGAS')
print('=' * 45)
for label, score in zip(labels, scores):
    bar = '█' * int(score * 20) + '░' * (20 - int(score * 20))
    print(f'{label.replace(chr(10)," "):20s} : {bar}  {score:.3f}')
print('=' * 45)
print(f'{"Score moyen":20s} : {np.mean(scores):.3f}')

# Graphique radar
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Évaluation RAGAS — BVMT Intelligence', fontsize=14, fontweight='bold')

# Bar chart
colors = ['#e94560' if s < 0.5 else '#f39c12' if s < 0.75 else '#27ae60' for s in scores]
bars = axes[0].bar(labels, scores, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_ylim(0, 1)
axes[0].set_title('Scores par métrique', fontweight='bold')
axes[0].axhline(y=0.7, color='gray', linestyle='--', alpha=0.5, label='Seuil qualité (0.7)')
axes[0].legend()
for bar, score in zip(bars, scores):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                 f'{score:.3f}', ha='center', va='bottom', fontweight='bold')

# Heatmap par question
heatmap_data = results_df[metrics].values
im = axes[1].imshow(heatmap_data, cmap='RdYlGn', vmin=0, vmax=1, aspect='auto')
axes[1].set_xticks(range(len(labels)))
axes[1].set_xticklabels(labels, fontsize=9)
axes[1].set_yticks(range(len(results_df)))
axes[1].set_yticklabels([f'Q{i+1}' for i in range(len(results_df))], fontsize=9)
axes[1].set_title('Heatmap par question', fontweight='bold')
plt.colorbar(im, ax=axes[1])
for i in range(len(results_df)):
    for j in range(len(metrics)):
        axes[1].text(j, i, f'{heatmap_data[i,j]:.2f}', ha='center', va='center',
                     fontsize=8, color='black', fontweight='bold')

plt.tight_layout()
plt.savefig('ragas_evaluation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n📁 Graphique sauvegardé dans ragas_evaluation_results.png')

## 8. Interprétation et recommandations

| Seuil | Interprétation |
|-------|----------------|
| > 0.75 | 🟢 Excellent |
| 0.50 – 0.75 | 🟡 Acceptable |
| < 0.50 | 🔴 À améliorer |

**Faithfulness** : Mesure si la réponse est ancrée dans les documents récupérés. Un score élevé indique peu ou pas d'hallucinations.

**Answer Relevancy** : Mesure si la réponse répond bien à la question posée. Un score faible indique des réponses hors sujet ou trop vagues.

**Context Precision** : Mesure si le contexte récupéré est pertinent. Un score faible indique que le retriever ramène trop de bruit.

**Context Recall** : Mesure si le contexte couvre bien la ground truth. Un score faible indique que des informations importantes ne sont pas récupérées.